In [20]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from zhipuai_embedding import ZhipuAIEmbeddings

_ = load_dotenv(find_dotenv())
zhipuai_api_key = os.environ.get("ZHIPUAI_API_KEY")

# ---------- 1. 收集文件路径 ----------
file_paths = []
folder_path = "./llm-universe/data_base/knowledge_db"
for root, dirs, files in os.walk(folder_path):
    for file in files:
        file_paths.append(os.path.join(root, file))
print(file_paths)

# ---------- 2. 统一加载成 Document 列表 ----------
docs = []
for file in file_paths:
    file_type = file.split(".")[-1].lower()
    if file_type == "pdf":
        docs.extend(PyMuPDFLoader(file).load())          # 自带 metadata["source"]
    elif file_type == "md":
        with open(file, "r", encoding="utf-8") as f:
            content = f.read()
        docs.append(Document(page_content=content, metadata={"source": file}))

# ---------- 3. 按标题分割（md 才有效果，pdf 无 # 标题会自动跳过）----------
headers_to_split_on = [
    ("#", "一级标题"),
    ("##", "二级标题"),
    ("###", "三级标题"),
]
header_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)

all_docs = []
for doc in docs:
    header_docs = header_splitter.split_text(doc.page_content)
    for hd in header_docs:
        # 继承来源路径（若原文有，用原文；否则用当前 doc 的）
        hd.metadata["source"] = doc.metadata.get("source", "unknown")
        all_docs.append(hd)

# ---------- 4. 递归字符分割（一次性）----------
CHUNK_SIZE = 500
OVERLAP_SIZE = 50
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE, chunk_overlap=OVERLAP_SIZE
)

final_docs = text_splitter.split_documents(all_docs)   # 返回扁平列表

# ---------- 5. 一次性存入向量库 ----------
embedding = ZhipuAIEmbeddings()
persist_directory = "./vector_db/chroma"

# vectordb = Chroma.from_documents(
#     documents=final_docs,
#     embedding=embedding,
#     persist_directory=persist_directory,
# )

['./llm-universe/data_base/knowledge_db/easy_rl/强化学习入门指南.txt', './llm-universe/data_base/knowledge_db/easy_rl/强化学习入门指南.tsv', './llm-universe/data_base/knowledge_db/easy_rl/强化学习入门指南.vtt', './llm-universe/data_base/knowledge_db/easy_rl/强化学习入门指南.json', './llm-universe/data_base/knowledge_db/easy_rl/强化学习入门指南.mp4', './llm-universe/data_base/knowledge_db/easy_rl/强化学习入门指南.srt', './llm-universe/data_base/knowledge_db/pumkin_book/pumpkin_book.pdf', './llm-universe/data_base/knowledge_db/prompt_engineering/4. 文本概括 Summarizing.md', './llm-universe/data_base/knowledge_db/prompt_engineering/2. 提示原则 Guidelines.md', './llm-universe/data_base/knowledge_db/prompt_engineering/6. 文本转换 Transforming.md', './llm-universe/data_base/knowledge_db/prompt_engineering/3. 迭代优化 Iterative.md', './llm-universe/data_base/knowledge_db/prompt_engineering/9. 总结 Summary.md', './llm-universe/data_base/knowledge_db/prompt_engineering/7. 文本扩展 Expanding.md', './llm-universe/data_base/knowledge_db/prompt_engineering/8. 聊天机器人 C

In [ ]:
print("文件总数:", len(file_paths))
print("加载文档数:", len(docs))
print("标题切分后块数:", len(all_docs))
print("递归切分后块数:", len(final_docs))

In [12]:
#加载向量数据库
vectordb = Chroma(
    embedding_function=embedding,
    persist_directory=persist_directory
)
vectordb
print(f"向量库中存储的数量：{vectordb._collection.count()}")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


向量库中存储的数量：1047


In [ ]:
question = "什么是prompt engineering?"
retriever = vectordb.as_retriever(search_kwargs={"k": 3})
docs = retriever.invoke(question)
print(f"检索到的内容数：{len(docs)}")


检索到的内容数：3


In [35]:
print(docs[1])

page_content='前言
“周志华老师的《机器学习》（西瓜书）是机器学习领域的经典入门教材之一，周老师为了使尽可能多的读
者通过西瓜书对机器学习有所了解, 所以在书中对部分公式的推导细节没有详述，但是这对那些想深究公式推
导细节的读者来说可能“不太友好”，本书旨在对西瓜书里比较难理解的公式加以解析，以及对部分公式补充
具体的推导细节。”
读到这里，大家可能会疑问为啥前面这段话加了引号，因为这只是我们最初的遐想，后来我们了解到，周
老师之所以省去这些推导细节的真实原因是，他本尊认为“理工科数学基础扎实点的大二下学生应该对西瓜书
中的推导细节无困难吧，要点在书里都有了，略去的细节应能脑补或做练习”。所以...... 本南瓜书只能算是我
等数学渣渣在自学的时候记下来的笔记，希望能够帮助大家都成为一名合格的“理工科数学基础扎实点的大二
下学生”。
使用说明
• 南瓜书的所有内容都是以西瓜书的内容为前置知识进行表述的，所以南瓜书的最佳使用方法是以西瓜书
为主线，遇到自己推导不出来或者看不懂的公式时再来查阅南瓜书；
• 对于初学机器学习的小白，西瓜书第1 章和第2 章的公式强烈不建议深究，简单过一下即可，等你学得
有点飘的时候再回来啃都来得及；
• 每个公式的解析和推导我们都力(zhi) 争(neng) 以本科数学基础的视角进行讲解，所以超纲的数学知识
我们通常都会以附录和参考文献的形式给出，感兴趣的同学可以继续沿着我们给的资料进行深入学习；
• 若南瓜书里没有你想要查阅的公式，或者你发现南瓜书哪个地方有错误，请毫不犹豫地去我们GitHub 的
Issues（地址：https://github.com/datawhalechina/pumpkin-book/issues）进行反馈，在对应版块
提交你希望补充的公式编号或者勘误信息，我们通常会在24 小时以内给您回复，超过24 小时未回复的
话可以微信联系我们（微信号：at-Sm1les）；
配套视频教程：https://www.bilibili.com/video/BV1Mh411e7VU
在线阅读地址：https://datawhalechina.github.io/pumpkin-book（仅供第1 版）
最新版PDF 获取地址：https://github.com/datawhalechina/pumpkin-bo

In [14]:
for i, doc in enumerate(docs):
    print(f"检索到的第{i+1}个内容: \n {doc.page_content}", end="\n-----------------------------------------------------\n")


检索到的第1个内容: 
 欢迎来到**面向开发者的提示工程**部分，本部分内容基于**吴恩达老师的《Prompt Engineering for Developer》课程**进行编写。《Prompt Engineering for Developer》课程是由**吴恩达老师**与 OpenAI 技术团队成员 **Isa Fulford** 老师合作授课，Isa 老师曾开发过受欢迎的 ChatGPT 检索插件，并且在教授 LLM （Large Language Model， 大语言模型）技术在产品中的应用方面做出了很大贡献。她还参与编写了教授人们使用 Prompt 的 OpenAI cookbook。我们希望通过本模块的学习，与大家分享使用提示词开发 LLM 应用的最佳实践和技巧。
-----------------------------------------------------
检索到的第2个内容: 
 网络上有许多关于提示词（Prompt， 本教程中将保留该术语）设计的材料，例如《30 prompts everyone has to know》之类的文章，这些文章主要集中在 **ChatGPT 的 Web 界面上**，许多人在使用它执行特定的、通常是一次性的任务。但我们认为，对于开发人员，**大语言模型（LLM） 的更强大功能是能通过 API 接口调用，从而快速构建软件应用程序**。实际上，我们了解到 DeepLearning.AI 的姊妹公司 AI Fund 的团队一直在与许多初创公司合作，将这些技术应用于诸多应用程序上。很兴奋能看到 LLM API 能够让开发人员非常快速地构建应用程序。  
在本模块，我们将与读者分享提升大语言模型应用效果的各种技巧和最佳实践。书中内容涵盖广泛，包括软件开发提示词设计、文本总结、推理、转换、扩展以及构建聊天机器人等语言模型典型应用场景。我们衷心希望该课程能激发读者的想象力，开发出更出色的语言模型应用。
-----------------------------------------------------
检索到的第3个内容: 
 在这一章中，我们将通过一个故事，引领你了解如何从产品评价和新闻文章中推导出情感和主题。  
让我们先想象一下，你是一名初创公司的数据分析师，你的任务是从各种产品评论和新闻文章中提取出关键的

In [ ]:
# 创建检索链
from langchain_core.runnables import RunnableLambda
def combine_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

combiner = RunnableLambda(combine_docs)
retrieval_chain = retriever | combiner

retrieval_chain.invoke("南瓜书是什么？")


'前言\n“周志华老师的《机器学习》（西瓜书）是机器学习领域的经典入门教材之一，周老师为了使尽可能多的读\n者通过西瓜书对机器学习有所了解, 所以在书中对部分公式的推导细节没有详述，但是这对那些想深究公式推\n导细节的读者来说可能“不太友好”，本书旨在对西瓜书里比较难理解的公式加以解析，以及对部分公式补充\n具体的推导细节。”\n读到这里，大家可能会疑问为啥前面这段话加了引号，因为这只是我们最初的遐想，后来我们了解到，周\n老师之所以省去这些推导细节的真实原因是，他本尊认为“理工科数学基础扎实点的大二下学生应该对西瓜书\n中的推导细节无困难吧，要点在书里都有了，略去的细节应能脑补或做练习”。所以...... 本南瓜书只能算是我\n等数学渣渣在自学的时候记下来的笔记，希望能够帮助大家都成为一名合格的“理工科数学基础扎实点的大二\n下学生”。\n使用说明\n• 南瓜书的所有内容都是以西瓜书的内容为前置知识进行表述的，所以南瓜书的最佳使用方法是以西瓜书\n为主线，遇到自己推导不出来或者看不懂的公式时再来查阅南瓜书；\n• 对于初学机器学习的小白，西瓜书第1 章和第2 章的公式强烈不建议深究，简单过一下即可，等你学得\n\n最新版PDF 获取地址：https://github.com/datawhalechina/pumpkin-book/releases\n编委会\n主编：Sm1les、archwalker、jbb0523\n编委：juxiao、Majingmin、MrBigFan、shanry、Ye980226\n封面设计：构思-Sm1les、创作-林王茂盛\n致谢\n特别感谢awyd234、feijuan、Ggmatch、Heitao5200、huaqing89、LongJH、LilRachel、LeoLRH、Nono17、\nspareribs、sunchaothu、StevenLzq 在最早期的时候对南瓜书所做的贡献。\n扫描下方二维码，然后回复关键词“南瓜书”，即可加入“南瓜书读者交流群”\n版权声明\n本作品采用知识共享署名-非商业性使用-相同方式共享4.0 国际许可协议进行许可。\n\n本:1.9.9\n发布日期:2023.03\n南  ⽠  书\nPUMPKIN\nB  O  O  K\nDatawhale'

In [26]:
#创建 LLM
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="glm-4.5-air",temperature=0.0,api_key=zhipuai_api_key,base_url="https://open.bigmodel.cn/api/paas/v4/")
llm.invoke("请简要介绍一下你自己，并且告诉我你当前的知识库中主要是什么方面的知识").content


'我是智谱AI训练的GLM大语言模型，旨在通过自然语言处理技术为用户提供信息查询、问题解答和创意支持等服务。我基于大规模文本数据训练，能够理解和生成人类语言，帮助用户完成各类知识性任务。\n\n我的知识库覆盖了广泛的主题，包括自然科学（物理、化学、生物）、技术（计算机、工程、医学）、人文社科（历史、文学、哲学）、日常生活常识（健康、教育、法律基础）等。知识主要来源于公开可用的文本资料，涵盖学术文献、百科全书、新闻报道、技术文档等，时效性截至2023年10月。\n\n需要注意的是，我的知识存在一定局限性，可能无法覆盖最新进展或高度专业化的领域，也不具备实时信息获取能力。对于重要决策（如医疗、法律等），建议咨询专业人士。有什么具体问题我可以帮你解答吗？'

In [27]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

template = """使用以下上下文来回答最后的问题。如果你不知道答案，就说你不知道，不要试图编造答
案。最多使用三句话。尽量使答案简明扼要。请你在回答的最后说“谢谢你的提问！”。
{context}
问题: {input}
"""
# 将template通过 PromptTemplate 转为可以在LCEL中使用的类型
prompt = PromptTemplate(template=template)

qa_chain = (
    RunnableParallel({"context": retrieval_chain, "input": RunnablePassthrough()})
    | prompt
    | llm
    | StrOutputParser()
)


In [30]:
question_1 = "什么是南瓜书？"
question_2 = "Prompt Engineering for Developer是谁写的？"

result = qa_chain.invoke(question_1)
print("大模型+知识库后回答 question_1 的结果：")
print(result)

print("\n==================\n")

result = qa_chain.invoke(question_2)
print("大模型+知识库后回答 question_2 的结果：")
print(result)

print("\n==================\n")

print(llm.invoke(question_1).content)

print("\n==================\n")

print(llm.invoke(question_2).content)



大模型+知识库后回答 question_1 的结果：
南瓜书是对周志华《机器学习》（西瓜书）的补充解析，旨在帮助读者理解书中难懂的公式推导细节。它由DataWhale China编委会编写，以本科数学基础视角讲解，适合自学时查阅。谢谢你的提问！


大模型+知识库后回答 question_2 的结果：
《Prompt Engineering for Developer》课程是由吴恩达老师和Isa Fulford老师合作编写的。吴恩达是课程的主要讲师，Isa Fulford是OpenAI技术团队成员。谢谢你的提问！


“南瓜书”这个称呼通常指的是**计算机视觉领域的一本经典教材**，它的正式名称是 **《计算机视觉：算法与应用》（Computer Vision: Algorithms and Applications）**。

以下是关于“南瓜书”的详细解释：

1.  **正式名称与作者：**
    *   书名：**Computer Vision: Algorithms and Applications**
    *   作者：**Richard Szeliski**（理查德·泽尔西斯基）
    *   出版社：Springer（施普林格）

2.  **为什么叫“南瓜书”？**
    *   **封面颜色：** 这本书最显著的特征是其**封面是鲜艳的南瓜橙色**。在计算机视觉的学习者和从业者中，这种独特的颜色使得这本书非常容易识别，久而久之，“南瓜书”就成了它的昵称，广为流传。
    *   **与“西瓜书”的对应：** 在中文计算机学习圈里，另一本经典教材——周志华教授的《机器学习》（封面是西瓜绿色）被昵称为“西瓜书”。由于南瓜和西瓜都是水果（或蔬菜），且封面颜色鲜明，这种昵称的类比方式非常形象，也使得“南瓜书”这个称呼更加深入人心。

3.  **内容与特点：**
    *   **全面性：** 这本书是计算机视觉领域非常全面和权威的教材之一。它系统地介绍了计算机视觉的核心概念、算法和应用。
    *   **深度与广度：** 内容覆盖了从基础的图像处理、特征提取、图像分割、立体视觉、运动估计、三维重建，到高级的物体识别、场景理解、图像生成、多视图几何等众多主题。
    *   **理论与实践结合：** 书中不仅包含详细的数学推导和算法描述

In [31]:
#向检索链添加聊天记录
from langchain_core.prompts import ChatPromptTemplate

# 问答链的系统prompt
system_prompt = (
    "你是一个问答任务的助手。 "
    "请使用检索到的上下文片段回答这个问题。 "
    "如果你不知道答案就说不知道。 "
    "请使用简洁的话语回答用户。"
    "\n\n"
    "{context}"
)
# 制定prompt template
qa_prompt = ChatPromptTemplate(
    [
        ("system", system_prompt),
        ("placeholder", "{chat_history}"),
        ("human", "{input}"),
    ]
)


In [32]:
# 有历史记录
messages = qa_prompt.invoke(
    {
        "input": "你可以介绍一下他吗？",
        "chat_history": [
            ("human", "西瓜书是什么？"),
            ("ai", "西瓜书是指周志华老师的《机器学习》一书，是机器学习领域的经典入门教材之一。"),
        ],
        "context": ""
    }
)
for message in messages.messages:
    print(message.content)


你是一个问答任务的助手。 请使用检索到的上下文片段回答这个问题。 如果你不知道答案就说不知道。 请使用简洁的话语回答用户。


西瓜书是什么？
西瓜书是指周志华老师的《机器学习》一书，是机器学习领域的经典入门教材之一。
你可以介绍一下他吗？


In [33]:
# 无历史记录
messages = qa_prompt.invoke(
    {
        "input": "南瓜书是什么？",
        "chat_history": [],
        "context": ""
    }
)
for message in messages.messages:
    print(message.content)


你是一个问答任务的助手。 请使用检索到的上下文片段回答这个问题。 如果你不知道答案就说不知道。 请使用简洁的话语回答用户。


南瓜书是什么？


In [34]:
from langchain_core.runnables import RunnableBranch

# 压缩问题的系统 prompt
condense_question_system_template = (
    "请根据聊天记录完善用户最新的问题，"
    "如果用户最新的问题不需要完善则返回用户的问题。"
    )
# 构造 压缩问题的 prompt template
condense_question_prompt = ChatPromptTemplate([
        ("system", condense_question_system_template),
        ("placeholder", "{chat_history}"),
        ("human", "{input}"),
    ])
# 构造检索文档的链
# RunnableBranch 会根据条件选择要运行的分支
retrieve_docs = RunnableBranch(
    # 分支 1: 若聊天记录中没有 chat_history 则直接使用用户问题查询向量数据库
    (lambda x: not x.get("chat_history", False), (lambda x: x["input"]) | retriever, ),
    # 分支 2 : 若聊天记录中有 chat_history 则先让 llm 根据聊天记录完善问题再查询向量数据库
    condense_question_prompt | llm | StrOutputParser() | retriever,
)


In [36]:
# 重新定义 combine_docs
def combine_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs["context"]) # 将 docs 改为 docs["context"]
# 定义问答链
qa_chain = (
    RunnablePassthrough.assign(context=combine_docs) # 使用 combine_docs 函数整合 qa_prompt 中的 context
    | qa_prompt # 问答模板
    | llm
    | StrOutputParser() # 规定输出的格式为 str
)
# 定义带有历史记录的问答链
qa_history_chain = RunnablePassthrough.assign(
    context = (lambda x: x) | retrieve_docs # 将查询结果存为 content
    ).assign(answer=qa_chain) # 将最终结果存为 answer


In [37]:
# 不带聊天记录
qa_history_chain.invoke({
    "input": "西瓜书是什么？",
    "chat_history": []
})


{'input': '西瓜书是什么？',
 'chat_history': [],
 'context': [Document(metadata={'source': './llm-universe/data_base/knowledge_db/pumkin_book/pumpkin_book.pdf'}, page_content='前言\n“周志华老师的《机器学习》（西瓜书）是机器学习领域的经典入门教材之一，周老师为了使尽可能多的读\n者通过西瓜书对机器学习有所了解, 所以在书中对部分公式的推导细节没有详述，但是这对那些想深究公式推\n导细节的读者来说可能“不太友好”，本书旨在对西瓜书里比较难理解的公式加以解析，以及对部分公式补充\n具体的推导细节。”\n读到这里，大家可能会疑问为啥前面这段话加了引号，因为这只是我们最初的遐想，后来我们了解到，周\n老师之所以省去这些推导细节的真实原因是，他本尊认为“理工科数学基础扎实点的大二下学生应该对西瓜书\n中的推导细节无困难吧，要点在书里都有了，略去的细节应能脑补或做练习”。所以...... 本南瓜书只能算是我\n等数学渣渣在自学的时候记下来的笔记，希望能够帮助大家都成为一名合格的“理工科数学基础扎实点的大二\n下学生”。\n使用说明\n• 南瓜书的所有内容都是以西瓜书的内容为前置知识进行表述的，所以南瓜书的最佳使用方法是以西瓜书\n为主线，遇到自己推导不出来或者看不懂的公式时再来查阅南瓜书；\n• 对于初学机器学习的小白，西瓜书第1 章和第2 章的公式强烈不建议深究，简单过一下即可，等你学得'),
  Document(metadata={'source': './llm-universe/data_base/knowledge_db/pumkin_book/pumpkin_book.pdf'}, page_content='• 对于初学机器学习的小白，西瓜书第1 章和第2 章的公式强烈不建议深究，简单过一下即可，等你学得\n有点飘的时候再回来啃都来得及；\n• 每个公式的解析和推导我们都力(zhi) 争(neng) 以本科数学基础的视角进行讲解，所以超纲的数学知识\n我们通常都会以附录和参考文献的形式给出，感兴趣的同学可以继续沿着我们给的资料进行深入学习；\n• 若南瓜书里没有你想要查阅的公式，或者你发现南瓜书哪个地

In [38]:
# 带聊天记录
qa_history_chain.invoke({
    "input": "南瓜书跟它有什么关系？",
    "chat_history": [
        ("human", "西瓜书是什么？"),
        ("ai", "西瓜书是指周志华老师的《机器学习》一书，是机器学习领域的经典入门教材之一。"),
    ]
})


{'input': '南瓜书跟它有什么关系？',
 'chat_history': [('human', '西瓜书是什么？'),
  ('ai', '西瓜书是指周志华老师的《机器学习》一书，是机器学习领域的经典入门教材之一。')],
 'context': [Document(metadata={'source': './llm-universe/data_base/knowledge_db/pumkin_book/pumpkin_book.pdf'}, page_content='前言\n“周志华老师的《机器学习》（西瓜书）是机器学习领域的经典入门教材之一，周老师为了使尽可能多的读\n者通过西瓜书对机器学习有所了解, 所以在书中对部分公式的推导细节没有详述，但是这对那些想深究公式推\n导细节的读者来说可能“不太友好”，本书旨在对西瓜书里比较难理解的公式加以解析，以及对部分公式补充\n具体的推导细节。”\n读到这里，大家可能会疑问为啥前面这段话加了引号，因为这只是我们最初的遐想，后来我们了解到，周\n老师之所以省去这些推导细节的真实原因是，他本尊认为“理工科数学基础扎实点的大二下学生应该对西瓜书\n中的推导细节无困难吧，要点在书里都有了，略去的细节应能脑补或做练习”。所以...... 本南瓜书只能算是我\n等数学渣渣在自学的时候记下来的笔记，希望能够帮助大家都成为一名合格的“理工科数学基础扎实点的大二\n下学生”。\n使用说明\n• 南瓜书的所有内容都是以西瓜书的内容为前置知识进行表述的，所以南瓜书的最佳使用方法是以西瓜书\n为主线，遇到自己推导不出来或者看不懂的公式时再来查阅南瓜书；\n• 对于初学机器学习的小白，西瓜书第1 章和第2 章的公式强烈不建议深究，简单过一下即可，等你学得'),
  Document(metadata={'source': './llm-universe/data_base/knowledge_db/pumkin_book/pumpkin_book.pdf'}, page_content='• 对于初学机器学习的小白，西瓜书第1 章和第2 章的公式强烈不建议深究，简单过一下即可，等你学得\n有点飘的时候再回来啃都来得及；\n• 每个公式的解析和推导我们都力(zhi) 争(neng) 以本科数学基础的视角进行讲解，所以超纲的数学知识\n我